# Using Karaage from Python

This notebook runs Karaage on the test dataset: one article, then a batch, then reads the outputs.
The command line does the same with `karaage run`; see the README.

Before running it, install and configure Karaage as the README says. By default, Karaage uses the model `qwen/qwen3.5-9b` in LM Studio on this machine.
Open the notebook from the repo root with `uv run --with jupyter jupyter lab examples/using-karaage-from-python.ipynb`; paths below are relative to `examples/`.

In [ ]:
import json
from pathlib import Path

import karaage
from karaage.batch import read_documents, run_documents

DATASET = Path("../data/test-datasets/wikinews-piracy-30/articles.jsonl")
OUTPUT = Path("../temp/notebook")   # temp/ is ignored by git

## Set up

`setup()` loads the NLP models and connects to the LLM backend, once per session.
The backend and the profile come from `KARAAGE_*` environment variables or a `.env` file; keyword arguments override them, e.g. `karaage.setup(profile="my-profile.toml")`.

In [ ]:
resources = karaage.setup()

## Read the documents

A document has an `id` and a `text`, and optionally a `title` and a `published_at` date.
The test dataset calls the publication date `date`, so it is read as `published_at`.

In [ ]:
documents = read_documents(DATASET, {"published_at": "date"})
article = documents[18]   # article 19
print(f"{len(documents)} documents. Article {article['id']}: {article['title']} ({article['published_at']})")

## Process one document

`run()` never raises when a step fails: the output then has the status `failed`, with the error.

In [ ]:
result = karaage.run(article, resources)
data = result.data
print(data["status"], "|", data["main_point"])

Every extracted statement is in the output, rejected ones included.

In [ ]:
for s in data["statements"]:
    verdict = "kept" if s["valid"] else f"rejected: {s['rejection_reason']}"
    print(f"{s['id']} ({verdict}) {s['actor']}: {s['text_short']}")

`save()` writes the output as JSON, and the HTML report, which shows each step's result. Open the report in a browser.

In [ ]:
karaage.save(result, data_path=OUTPUT / "19.json", html_path=OUTPUT / "19.html")

## Process a batch

`run_documents()` does what `karaage run` does: one output per document, and a report with `report=True`.
Documents whose output is already `ok` are skipped, so running the cell again resumes an interrupted batch and retries the documents that failed.
Here, the first five articles.

In [ ]:
statuses = run_documents(documents[:5], OUTPUT, resources, report=True)

## Read the outputs

The output format is described in the README and by the JSON Schema in `src/karaage/schemas/`.

In [ ]:
for path in sorted(OUTPUT.glob("*.json")):
    output = json.loads(path.read_text(encoding="utf-8"))
    kept = sum(s["valid"] for s in output["statements"])
    print(f"Document {output['document_id']:>2}: {output['status']}, {len(output['actors'])} actors, "
          f"{kept} of {len(output['statements'])} statements kept")

## Optional: inspect a rephrasing batch

Statements are rephrased in batches of four. `batch_debug()` prints the prompt of one batch of the document processed above, and sends it again to show the model's raw answer.

In [ ]:
from karaage.rephrasing import batch_debug

batch_debug(result.all_statements, result.excerpts, data["main_point"], batch_index=0)